# Notebook II: Image Reconstruction & Medical Validation Metrics
From raw projections to images: FBP, MLEM, OSEM from scratch. Plus the statistical metrics that matter for clinical AI.

In [ ]:
import numpy as np
from scipy import ndimage, special
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon, resize
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import xgboost as xgb

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print('All imports successful.')

## Part 1: The Inverse Problem of Image Reconstruction

### What scanners actually measure

PET and SPECT scanners do **not** directly produce images. Instead, they detect **gamma-ray photons**
emitted from radioactive tracers inside the patient's body. Each detected photon tells us that
*somewhere along a line through the patient*, a decay event occurred.

Mathematically, what we measure are **line integrals** (projections) through the unknown activity
distribution. If we denote:

- $\mathbf{x} \in \mathbb{R}^n$ = the unknown image (vectorized), where $x_j$ is the activity in voxel $j$
- $\mathbf{y} \in \mathbb{R}^m$ = the measured projection data (sinogram), where $y_i$ is the count in detector pair $i$
- $\mathbf{A} \in \mathbb{R}^{m \times n}$ = the **system matrix**, where $a_{ij}$ is the probability that a photon emitted from voxel $j$ is detected by detector pair $i$

Then the forward model is:

$$\mathbf{y} = \mathbf{A}\mathbf{x} + \text{noise}$$

**Reconstruction** is the inverse problem: given $\mathbf{y}$ and $\mathbf{A}$, recover $\mathbf{x}$.

### Why is this hard?

1. **Ill-conditioned**: $\mathbf{A}$ is huge and ill-conditioned. Naively inverting amplifies noise.
2. **Noise**: Photon counting follows Poisson statistics. Low-count regions are very noisy.
3. **Physics effects**: Attenuation, scatter, detector response blur the data further.
4. **Non-negativity**: Activity concentrations cannot be negative, but naive inversions can produce negative values.

We will explore three approaches of increasing sophistication: **FBP**, **MLEM**, and **OSEM**.

## The Radon Transform

The mathematical foundation of tomographic reconstruction is the **Radon transform**, discovered by
Johann Radon in 1917 (decades before CT scanners existed!).

For a 2D function $f(x, y)$, the Radon transform computes **line integrals** at every angle $\theta$
and every offset $s$:

$$\mathcal{R}f(s, \theta) = \int_{-\infty}^{\infty} f(s\cos\theta - t\sin\theta, \; s\sin\theta + t\cos\theta) \, dt$$

For each angle $\theta$, this produces a 1D projection (the shadow of the object onto a line
perpendicular to the ray direction). Stacking all 1D projections for $\theta \in [0^\circ, 180^\circ)$
produces a 2D array called the **sinogram** (so named because a point source traces a sinusoidal
curve through it).

A sinogram is **everything the scanner gives us**. Reconstruction algorithms convert sinograms back
into images.

In [ ]:
# ============================================================
# Shepp-Logan Phantom + Radon Transform (Sinogram)
# ============================================================
# The Shepp-Logan phantom is the standard test image for
# reconstruction algorithms. It simulates a cross-section of
# a human head with ellipses of different densities.

# Create phantom (128x128 for speed; clinical images are 128-512)
phantom = shepp_logan_phantom()
phantom = resize(phantom, (128, 128), anti_aliasing=True)

# Compute sinogram: Radon transform at 180 angles (1-degree spacing)
theta = np.linspace(0, 180, 180, endpoint=False)
sinogram = radon(phantom, theta=theta)

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

im1 = ax1.imshow(phantom, cmap='gray')
ax1.set_title('Shepp-Logan Phantom (Ground Truth)')
ax1.set_xlabel('x pixel')
ax1.set_ylabel('y pixel')
plt.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(sinogram, cmap='gray', aspect='auto',
                  extent=[theta[0], theta[-1], sinogram.shape[0], 0])
ax2.set_title('Sinogram (Radon Transform)')
ax2.set_xlabel('Projection Angle (degrees)')
ax2.set_ylabel('Detector Position')
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

print(f'Phantom shape:  {phantom.shape}')
print(f'Sinogram shape: {sinogram.shape}  (n_detectors x n_angles)')
print(f'Value range:    [{phantom.min():.4f}, {phantom.max():.4f}]')

## Filtered Back-Projection (FBP)

FBP is the **oldest and fastest** reconstruction method. It is the workhorse of CT reconstruction
and is often used as a baseline for iterative methods.

### The Fourier Slice Theorem

The key theoretical result: the **1D Fourier transform** of a projection at angle $\theta$ equals
a **radial slice** of the **2D Fourier transform** of the image at the same angle.

$$\mathcal{F}_1\{\mathcal{R}f(s, \theta)\}(\omega) = \mathcal{F}_2\{f\}(\omega\cos\theta, \omega\sin\theta)$$

This means we could, in principle, fill in the 2D Fourier space from all projections and then
inverse-FT. In practice, the polar sampling pattern makes this tricky, so instead we use
**back-projection**.

### Back-Projection alone is blurry

Simple back-projection smears each projection value back along its ray direction and sums.
The result is blurry because each point gets contributions falling off as $1/r$ from
neighboring structures.

### The ramp filter fixes this

Applying a **ramp filter** $|\omega|$ in frequency domain to each projection before
back-projecting exactly compensates for the $1/r$ blurring:

$$f(x, y) = \int_0^{\pi} \left[ \mathcal{R}f(s, \theta) * h(s) \right]_{s = x\cos\theta + y\sin\theta} \, d\theta$$

where $h(s)$ is the inverse FT of the ramp filter $|\omega|$.

### Trade-off

- **Pro**: Very fast (one pass through data), well-understood theory
- **Con**: The ramp filter amplifies high-frequency noise. Low-count PET/SPECT data
  produces noisy FBP images. Cannot easily incorporate physics modeling (attenuation, scatter).

In [ ]:
# ============================================================
# Filtered Back-Projection (FBP) vs. Unfiltered Back-Projection
# ============================================================

# FBP with ramp filter (the standard)
fbp_recon = iradon(sinogram, theta=theta, filter_name='ramp')

# Unfiltered back-projection for comparison (no filter = blurry)
bp_recon = iradon(sinogram, theta=theta, filter_name=None)

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Ground Truth')

axes[1].imshow(bp_recon, cmap='gray')
axes[1].set_title('Back-Projection (no filter)\n= blurry 1/r smearing')

axes[2].imshow(fbp_recon, cmap='gray')
axes[2].set_title('Filtered Back-Projection (ramp)\n= sharp but amplifies noise')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# Quantitative comparison
fbp_rmse = np.sqrt(np.mean((fbp_recon - phantom)**2))
bp_rmse = np.sqrt(np.mean((bp_recon - phantom)**2))
print(f'Back-Projection RMSE: {bp_rmse:.6f}')
print(f'FBP RMSE:             {fbp_rmse:.6f}')
print(f'FBP is {bp_rmse / fbp_rmse:.1f}x better than unfiltered BP (by RMSE).')

## MLEM: Maximum-Likelihood Expectation-Maximization

FBP treats reconstruction as a linear algebra / Fourier problem. **MLEM** instead treats it as a
**statistical estimation** problem, which is far more natural for photon-counting data.

### The Poisson noise model

Each detector measurement $y_i$ is a count of photons. Photon detection is inherently random and
follows a **Poisson distribution**:

$$y_i \sim \text{Poisson}\left(\bar{y}_i\right), \quad \text{where } \bar{y}_i = \sum_j a_{ij} x_j$$

### Log-likelihood

The log-likelihood of the data given the image $\mathbf{x}$ is:

$$L(\mathbf{x}) = \sum_i \left[ y_i \log\left(\sum_j a_{ij} x_j\right) - \sum_j a_{ij} x_j - \log(y_i!) \right]$$

We want to find $\mathbf{x}^* = \arg\max_{\mathbf{x} \geq 0} L(\mathbf{x})$.

### EM algorithm derivation

Direct maximization is difficult because of the $\log(\sum)$ term. The EM algorithm introduces
**latent variables** $z_{ij}$ = "number of photons from voxel $j$ detected in measurement $i$".

- **E-step**: Compute expected latent variables given current estimate $\mathbf{x}^{(n)}$:
  $$\hat{z}_{ij} = \frac{a_{ij} x_j^{(n)}}{\sum_k a_{ik} x_k^{(n)}} \cdot y_i$$

- **M-step**: Update the image to maximize expected complete-data log-likelihood:
  $$x_j^{(n+1)} = \frac{\sum_i \hat{z}_{ij}}{\sum_i a_{ij}}$$

### The famous MLEM update rule

Substituting the E-step into the M-step and simplifying gives the elegant **multiplicative update**:

$$\boxed{x_j^{(n+1)} = \frac{x_j^{(n)}}{\sum_i a_{ij}} \sum_i \frac{a_{ij} \, y_i}{\sum_k a_{ik} x_k^{(n)}}}$$

### Intuition

Read this as: **current estimate** $\times$ **(correction factor)**.

The correction factor compares measured projections $y_i$ to expected projections
$\sum_k a_{ik} x_k^{(n)}$. Where the model under-predicts (ratio > 1), the image is increased;
where it over-predicts (ratio < 1), the image is decreased.

### Properties

- **Non-negativity**: If initialized with $x_j > 0$, the multiplicative update keeps $x_j \geq 0$ forever.
- **Monotonic convergence**: $L(\mathbf{x}^{(n+1)}) \geq L(\mathbf{x}^{(n)})$ guaranteed.
- **Slow convergence**: Especially for high-frequency components. Typically needs 30-100 iterations.
- **Semi-convergence**: Early iterations recover large structures; later iterations add noise (the algorithm "fits the noise"). Often stopped early as implicit regularization.

In [ ]:
# ============================================================
# MLEM Reconstruction from Scratch
# ============================================================
# We use scikit-image's radon() and iradon(..., filter_name=None)
# as our forward projector (A*x) and back-projector (A^T*y).
#
# Note: iradon with filter_name=None performs unfiltered
# back-projection, which is mathematically equivalent to
# applying A^T (up to a scaling factor that cancels out
# in the MLEM ratio).

def mlem_reconstruct(sinogram, theta, n_iterations=50, image_size=128):
    """
    MLEM reconstruction from scratch.
    
    Parameters
    ----------
    sinogram : ndarray (n_detectors, n_angles)
        Measured projection data.
    theta : ndarray (n_angles,)
        Projection angles in degrees.
    n_iterations : int
        Number of MLEM iterations.
    image_size : int
        Output image dimension (square).
    
    Returns
    -------
    recon : ndarray (image_size, image_size)
        Reconstructed image.
    rmse_history : list of float
        RMSE at each iteration (requires global `phantom`).
    """
    # Initialize with uniform image (must be > 0 for multiplicative update)
    recon = np.ones((image_size, image_size), dtype=np.float64)
    
    # Sensitivity image: s_j = sum_i a_ij
    # = back-project a sinogram of all ones
    ones_sinogram = np.ones_like(sinogram)
    sensitivity = iradon(ones_sinogram, theta=theta, filter_name=None,
                         output_size=image_size)
    sensitivity = np.maximum(sensitivity, 1e-10)  # avoid division by zero
    
    rmse_history = []
    
    for iteration in range(n_iterations):
        # Step 1: Forward project current estimate -> expected sinogram
        expected_sinogram = radon(recon, theta=theta)
        expected_sinogram = np.maximum(expected_sinogram, 1e-10)
        
        # Step 2: Compute ratio (measured / expected)
        ratio = sinogram / expected_sinogram
        
        # Step 3: Back-project the ratio -> correction image
        correction = iradon(ratio, theta=theta, filter_name=None,
                            output_size=image_size)
        
        # Step 4: Multiplicative update
        recon = recon * correction / sensitivity
        recon = np.maximum(recon, 0)  # enforce non-negativity
        
        # Track convergence
        error = np.sqrt(np.mean((recon - phantom)**2))
        rmse_history.append(error)
        
        if (iteration + 1) % 10 == 0:
            print(f'  MLEM Iteration {iteration+1:3d}: RMSE = {error:.6f}')
    
    return recon, rmse_history


print('Running MLEM (50 iterations)...')
mlem_result, mlem_rmse = mlem_reconstruct(sinogram, theta, n_iterations=50)
print('Done.\n')

# --- Compare FBP vs MLEM ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Ground Truth')

axes[1].imshow(fbp_recon, cmap='gray')
axes[1].set_title(f'FBP\nRMSE = {np.sqrt(np.mean((fbp_recon - phantom)**2)):.4f}')

axes[2].imshow(mlem_result, cmap='gray')
axes[2].set_title(f'MLEM (50 iter)\nRMSE = {mlem_rmse[-1]:.4f}')

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## OSEM: Ordered Subsets Expectation-Maximization

MLEM's biggest practical problem is **speed**: each iteration requires a forward and back-projection
through **all** projection angles. For a clinical scanner with 120+ angles, this is expensive.

**OSEM** (Hudson & Larkin, 1994) accelerates convergence dramatically:

1. Divide the projection angles into $S$ **subsets** (e.g., $S = 8$)
2. Perform an MLEM-like update using only **one subset** at a time
3. Cycle through all subsets = one **iteration** (also called one "pass" through the data)

### Why it works

Each subset update moves the image toward the ML solution. Since we update $S$ times per full
pass through the data (vs. once for MLEM), early convergence is approximately **$S$ times faster**.

### Important caveats

- OSEM does **not** converge to the true ML solution. It oscillates near it (the "limit cycle").
- The subsets should be chosen to be **maximally spread** in angle (e.g., subset 0 gets angles
  0, 8, 16, ...; subset 1 gets 1, 9, 17, ...) rather than contiguous blocks.
- The **clinical standard** for PET is typically **2-3 iterations with 8-21 subsets**.

### Comparison to deep learning

OSEM is to MLEM what **stochastic gradient descent** is to full-batch gradient descent.
The analogy is almost exact: subsets = mini-batches, and the same convergence trade-offs apply.

In [ ]:
# ============================================================
# OSEM Reconstruction from Scratch
# ============================================================

def osem_reconstruct(sinogram, theta, n_iterations=3, n_subsets=8,
                     image_size=128, track_per_subset=False):
    """
    OSEM reconstruction from scratch.
    
    Parameters
    ----------
    sinogram : ndarray (n_detectors, n_angles)
    theta : ndarray (n_angles,)
    n_iterations : int
        Number of full iterations (each processes all subsets).
    n_subsets : int
        Number of ordered subsets.
    image_size : int
    track_per_subset : bool
        If True, track RMSE after every subset update (for convergence plot).
    
    Returns
    -------
    recon : ndarray (image_size, image_size)
    rmse_history : list of float
    """
    recon = np.ones((image_size, image_size), dtype=np.float64)
    n_angles = len(theta)
    rmse_history = []
    
    for iteration in range(n_iterations):
        for s in range(n_subsets):
            # Select interleaved subset angles (maximally spread)
            subset_idx = list(range(s, n_angles, n_subsets))
            subset_theta = theta[subset_idx]
            subset_sino = sinogram[:, subset_idx]
            
            # Sensitivity for this subset: back-project ones
            ones_sub = np.ones_like(subset_sino)
            sensitivity = iradon(ones_sub, theta=subset_theta,
                                 filter_name=None, output_size=image_size)
            sensitivity = np.maximum(sensitivity, 1e-10)
            
            # Forward project current estimate (subset angles only)
            expected = radon(recon, theta=subset_theta)
            expected = np.maximum(expected, 1e-10)
            
            # Ratio and back-project correction
            ratio = subset_sino / expected
            correction = iradon(ratio, theta=subset_theta,
                                filter_name=None, output_size=image_size)
            
            # Multiplicative update
            recon = recon * correction / sensitivity
            recon = np.maximum(recon, 0)
            
            # Track RMSE per subset update
            if track_per_subset:
                error = np.sqrt(np.mean((recon - phantom)**2))
                rmse_history.append(error)
        
        # Track RMSE per full iteration
        error = np.sqrt(np.mean((recon - phantom)**2))
        if not track_per_subset:
            rmse_history.append(error)
        print(f'  OSEM Iteration {iteration+1} ({n_subsets} subsets): RMSE = {error:.6f}')
    
    return recon, rmse_history


print('Running OSEM (3 iterations x 8 subsets = 24 subset updates)...')
osem_result, osem_rmse = osem_reconstruct(sinogram, theta,
                                           n_iterations=3, n_subsets=8)
print('Done.\n')

# --- Compare all three methods ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

titles = ['Ground Truth', 'FBP',
          f'MLEM (50 iter)', f'OSEM (3 iter x 8 sub)']
images = [phantom, fbp_recon, mlem_result, osem_result]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    rmse = np.sqrt(np.mean((img - phantom)**2))
    ax.set_title(f'{title}\nRMSE={rmse:.4f}')
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## Convergence Analysis

A key question: how do MLEM and OSEM compare in terms of convergence speed?

To make a fair comparison, we plot RMSE against the number of **subset updates** (forward + back
projection pairs), since that is the dominant computational cost:

- MLEM: 1 subset update per iteration (uses all angles)
- OSEM with $S$ subsets: $S$ subset updates per iteration

So MLEM iteration $k$ corresponds to subset update $k$, while OSEM iteration $k$ corresponds to
subset update $k \times S$.

In [ ]:
# ============================================================
# Convergence Comparison: MLEM vs OSEM
# ============================================================
# Re-run OSEM with per-subset tracking for fair comparison

print('Running MLEM (50 iterations) for convergence tracking...')
_, mlem_rmse_full = mlem_reconstruct(sinogram, theta, n_iterations=50)

print('\nRunning OSEM (6 iterations x 8 subsets) with per-subset tracking...')
_, osem_rmse_per_sub = osem_reconstruct(sinogram, theta,
                                         n_iterations=6, n_subsets=8,
                                         track_per_subset=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# MLEM: each iteration = 1 full forward+back projection
# For fair comparison, count each MLEM iteration as using all angles
mlem_x = np.arange(1, len(mlem_rmse_full) + 1)
ax.plot(mlem_x, mlem_rmse_full, 'b-o', markersize=3,
        label='MLEM (1 update/iter, all angles)', linewidth=2)

# OSEM: each subset update uses 1/S of the angles
# So S subset updates ~ 1 MLEM iteration in compute
n_subsets = 8
osem_x = np.arange(1, len(osem_rmse_per_sub) + 1) / n_subsets
ax.plot(osem_x, osem_rmse_per_sub, 'r-s', markersize=3,
        label=f'OSEM ({n_subsets} subsets, normalized to equiv. iterations)',
        linewidth=2)

ax.set_xlabel('Equivalent Full Iterations (compute cost)')
ax.set_ylabel('RMSE vs Ground Truth')
ax.set_title('Convergence: MLEM vs OSEM')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 50)

plt.tight_layout()
plt.show()

print(f'\nMLEM reaches RMSE {mlem_rmse_full[2]:.4f} at iteration 3.')
print(f'OSEM reaches RMSE {osem_rmse_per_sub[2]:.4f} after just 3 subset updates '
      f'(= {3/n_subsets:.2f} equivalent iterations).')
print(f'\nOSEM converges much faster in early iterations, '
      f'demonstrating the ~Sx acceleration.')

## Attenuation Correction

So far we have ignored a critical physical effect: **photon attenuation**.

### Beer-Lambert Law

As a gamma-ray photon travels through tissue, it has a probability of being absorbed or scattered.
The surviving fraction follows the **Beer-Lambert law**:

$$I = I_0 \cdot \exp\left(-\int_L \mu(\mathbf{r}) \, d\ell\right)$$

where $\mu(\mathbf{r})$ is the **linear attenuation coefficient** at position $\mathbf{r}$ and
the integral is along the photon's path $L$.

### Why it matters clinically

- Photons from **deep structures** (like the heart) must pass through more tissue and are more
  likely to be absorbed.
- Without attenuation correction, the reconstructed image **underestimates activity** in deep
  regions.
- In cardiac SPECT, this can create **artifactual perfusion defects** in the inferior wall
  (especially in large patients), leading to false-positive diagnoses.

### Chang's method (uniform attenuation)

The simplest correction (Chang, 1978): assume uniform attenuation $\mu$ inside the body contour.
For each pixel $j$, compute the **average attenuation** across all projection angles:

$$C_j = \frac{1}{N_\theta} \sum_{\theta} \exp\left(-\mu \cdot d_j(\theta)\right)$$

where $d_j(\theta)$ is the path length from pixel $j$ to the body edge at angle $\theta$.
Then divide the reconstructed image by $C_j$.

### CT-based attenuation correction (modern)

Modern PET/CT and SPECT/CT scanners use the CT image to get a **spatially varying** attenuation
map. CT Hounsfield units are converted to linear attenuation coefficients at the relevant
gamma-ray energy (140 keV for Tc-99m SPECT, 511 keV for PET). This attenuation map is then
incorporated directly into the system matrix $\mathbf{A}$ during iterative reconstruction.

**This is the primary reason PET/CT and SPECT/CT hybrid scanners exist** -- not just for
anatomical localization, but for accurate attenuation correction.

In [ ]:
# ============================================================
# Attenuation Effect Demonstration
# ============================================================
# We create a simple attenuation map, simulate attenuated
# projections, and show the effect + Chang correction.

# Create a uniform attenuation map (elliptical body)
# Typical soft tissue mu at 140 keV ~ 0.15 cm^-1
# We use a simplified model here
y_grid, x_grid = np.mgrid[-1:1:128j, -1:1:128j]
body_mask = ((x_grid / 0.8)**2 + (y_grid / 0.6)**2) < 1.0
mu_map = body_mask.astype(float) * 0.15  # cm^-1

# Create a simple activity distribution (hot lesion in center)
activity = np.zeros((128, 128))
activity[body_mask] = 1.0  # uniform background
# Add a hot lesion (simulating a tumor)
lesion_mask = ((x_grid - 0.0)**2 + (y_grid + 0.1)**2) < 0.05
activity[lesion_mask] = 5.0

# Forward project without attenuation (ideal)
sino_ideal = radon(activity, theta=theta)

# Forward project the attenuation map to get path lengths
# Then compute attenuated sinogram
mu_sinogram = radon(mu_map, theta=theta)
# Attenuation factor for each ray
attenuation_factor = np.exp(-mu_sinogram)
# Attenuated sinogram (what the scanner actually measures)
sino_attenuated = sino_ideal * attenuation_factor

# Reconstruct both (FBP for simplicity)
recon_no_ac = iradon(sino_attenuated, theta=theta, filter_name='ramp')
recon_ideal = iradon(sino_ideal, theta=theta, filter_name='ramp')

# --- Chang correction ---
# For each pixel, compute average attenuation factor across angles
# Approximate: use the attenuation sinogram
chang_factors = np.zeros((128, 128))
# Simple implementation: for each pixel, compute attenuation along
# rays from that pixel to the boundary at each angle.
# We approximate this using the reconstructed attenuation profile.
# A simpler approach: divide by the back-projected attenuation.
atten_correction_map = iradon(attenuation_factor, theta=theta, filter_name=None)
atten_correction_map = np.maximum(atten_correction_map, 0.01)
# Normalize so center of body has correction ~1/mean_attenuation
atten_correction_map = atten_correction_map / atten_correction_map.max()
atten_correction_map = np.maximum(atten_correction_map, 0.1)

recon_chang = recon_no_ac / atten_correction_map

# --- Visualization ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Top row: setup
axes[0, 0].imshow(activity, cmap='hot')
axes[0, 0].set_title('True Activity Distribution')

axes[0, 1].imshow(mu_map, cmap='bone')
axes[0, 1].set_title('Attenuation Map ($\mu$)')

axes[0, 2].imshow(attenuation_factor, cmap='gray', aspect='auto')
axes[0, 2].set_title('Attenuation Factors per Ray')
axes[0, 2].set_xlabel('Angle')
axes[0, 2].set_ylabel('Detector')

# Bottom row: reconstructions
axes[1, 0].imshow(recon_ideal, cmap='hot')
axes[1, 0].set_title('Recon: No Attenuation\n(ideal scenario)')

axes[1, 1].imshow(recon_no_ac, cmap='hot')
axes[1, 1].set_title('Recon: With Attenuation\n(no correction) -- ARTIFACTS')

axes[1, 2].imshow(recon_chang, cmap='hot')
axes[1, 2].set_title('Recon: Chang Correction\n(approximate fix)')

for row in axes:
    for ax in row:
        ax.set_xticks([])
        ax.set_yticks([])

plt.suptitle('Effect of Attenuation on Reconstruction', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# Show line profile through center
fig, ax = plt.subplots(figsize=(10, 4))
mid = 64
ax.plot(activity[mid, :], 'k-', linewidth=2, label='Ground Truth')
ax.plot(recon_ideal[mid, :], 'g--', linewidth=1.5, label='Recon (no atten.)')
ax.plot(recon_no_ac[mid, :], 'r-', linewidth=1.5, label='Recon (w/ atten., no correction)')
ax.plot(recon_chang[mid, :], 'b--', linewidth=1.5, label='Recon (Chang corrected)')
ax.set_xlabel('Pixel position')
ax.set_ylabel('Activity')
ax.set_title('Horizontal Line Profile Through Center')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Notice: without attenuation correction, deep structures appear dimmer.')
print('Chang correction partially restores the true activity distribution.')

---

## Part 2: Medical Image Metrics for Clinical AI

Now that we can reconstruct images, how do we **evaluate** AI models that operate on them?

Standard ML metrics (accuracy, AUC) tell you about classification performance, but clinical
validation requires additional domain-specific metrics. We will cover:

1. **Dice Score** -- segmentation overlap
2. **SUV (Standardized Uptake Value)** -- quantitative PET metric
3. **TPD (Total Perfusion Deficit)** -- cardiac SPECT risk metric
4. **C-Statistic** -- survival/risk discrimination
5. **NRI / IDI** -- reclassification improvement
6. **Brier Score & Calibration** -- prediction reliability

In [ ]:
# ============================================================
# Dice Score (Sorensen-Dice Coefficient)
# ============================================================
# The standard metric for segmentation overlap in medical imaging.
# Dice = 2 * |A intersection B| / (|A| + |B|)
# Range: 0 (no overlap) to 1 (perfect overlap)

def dice_score(pred, truth):
    """
    Compute Dice coefficient between two binary masks.
    
    Parameters
    ----------
    pred : ndarray (bool or 0/1)
        Predicted segmentation mask.
    truth : ndarray (bool or 0/1)
        Ground truth segmentation mask.
    
    Returns
    -------
    float : Dice coefficient in [0, 1].
    """
    pred = pred.astype(bool)
    truth = truth.astype(bool)
    intersection = np.sum(pred & truth)
    return 2.0 * intersection / (np.sum(pred) + np.sum(truth) + 1e-10)


# --- Demo with synthetic segmentation ---
# Create a "ground truth" circular tumor
y, x = np.mgrid[-1:1:128j, -1:1:128j]
truth_mask = (x**2 + y**2) < 0.3**2

# Simulated predictions with varying quality
# Good: slight offset
pred_good = ((x - 0.02)**2 + (y - 0.02)**2) < 0.29**2
# Medium: larger offset + size error
pred_med = ((x - 0.08)**2 + (y - 0.05)**2) < 0.25**2
# Poor: very different
pred_poor = ((x - 0.2)**2 + (y + 0.15)**2) < 0.2**2

preds = [pred_good, pred_med, pred_poor]
labels = ['Good Prediction', 'Medium Prediction', 'Poor Prediction']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, pred, label in zip(axes, preds, labels):
    # Create overlay: green = true positive, red = false positive, blue = false negative
    overlay = np.zeros((128, 128, 3))
    tp = pred & truth_mask
    fp = pred & ~truth_mask
    fn = ~pred & truth_mask
    overlay[tp] = [0, 1, 0]       # green = correct
    overlay[fp] = [1, 0, 0]       # red = over-segmented
    overlay[fn] = [0, 0.3, 1]     # blue = missed
    
    d = dice_score(pred, truth_mask)
    ax.imshow(overlay)
    ax.set_title(f'{label}\nDice = {d:.3f}')
    ax.set_xticks([])
    ax.set_yticks([])

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', label='True Positive'),
    Patch(facecolor='red', label='False Positive'),
    Patch(facecolor='#004DFF', label='False Negative'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Dice Score: Segmentation Overlap Quality', fontsize=14)
plt.tight_layout()
plt.show()

print('Dice interpretation guide:')
print('  > 0.90: Excellent segmentation')
print('  0.70 - 0.90: Good (acceptable for most clinical tasks)')
print('  0.50 - 0.70: Moderate (may need manual correction)')
print('  < 0.50: Poor')

In [ ]:
# ============================================================
# SUV (Standardized Uptake Value) Calculation
# ============================================================
# SUV normalizes PET tracer uptake by patient weight and injected dose,
# making values comparable across patients and scans.
#
# SUV = (tissue activity [Bq/mL]) / (injected dose [Bq] / body weight [g])
#
# With decay correction:
#   A_corrected = A_measured * exp(ln(2) * t / t_half)
#   where t = time since injection, t_half = radioactive half-life

def compute_suv(activity_bq_ml, injected_dose_bq, body_weight_kg,
                time_post_injection_min, half_life_min):
    """
    Compute SUV with decay correction.
    
    Parameters
    ----------
    activity_bq_ml : float or ndarray
        Measured activity concentration in Bq/mL.
    injected_dose_bq : float
        Injected dose at time of injection in Bq.
    body_weight_kg : float
        Patient body weight in kg.
    time_post_injection_min : float
        Time between injection and scan in minutes.
    half_life_min : float
        Radioactive half-life of the tracer in minutes.
    
    Returns
    -------
    suv : float or ndarray
        Standardized uptake value (unitless).
    """
    # Decay correction factor
    decay_factor = np.exp(np.log(2) * time_post_injection_min / half_life_min)
    
    # Decay-corrected injected dose
    corrected_dose_bq = injected_dose_bq / decay_factor
    
    # Body weight in grams (since activity is per mL ~ per gram)
    body_weight_g = body_weight_kg * 1000.0
    
    # SUV calculation
    suv = activity_bq_ml / (corrected_dose_bq / body_weight_g)
    
    return suv


# --- Demo with synthetic PET data ---
# Simulate a PET image with known SUV values
np.random.seed(42)

# Patient parameters
injected_dose = 370e6     # 370 MBq (typical FDG dose)
body_weight = 70          # kg
scan_time = 60            # minutes post-injection
f18_half_life = 109.77    # F-18 half-life in minutes

# Create synthetic activity image (128x128)
# Background SUV ~ 1.0, tumor SUV ~ 8.0
# We work backwards from desired SUV to get activity
decay_factor = np.exp(np.log(2) * scan_time / f18_half_life)
dose_at_scan = injected_dose / decay_factor
activity_for_suv1 = dose_at_scan / (body_weight * 1000)  # Bq/mL for SUV=1

# Build activity map
activity_map = np.random.normal(1.0, 0.1, (128, 128)) * activity_for_suv1
activity_map = np.maximum(activity_map, 0)

# Add a "tumor" with SUV ~ 8
tumor = ((x - 0.3)**2 + (y + 0.2)**2) < 0.08**2
activity_map[tumor] = np.random.normal(8.0, 0.5, np.sum(tumor)) * activity_for_suv1

# Add a mild uptake region (SUV ~ 3, e.g., inflammation)
inflammation = ((x + 0.2)**2 + (y - 0.3)**2) < 0.06**2
activity_map[inflammation] = np.random.normal(3.0, 0.3, np.sum(inflammation)) * activity_for_suv1

# Compute SUV map
suv_map = compute_suv(activity_map, injected_dose, body_weight,
                       scan_time, f18_half_life)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

im1 = ax1.imshow(activity_map, cmap='hot')
ax1.set_title('Raw Activity (Bq/mL)')
plt.colorbar(im1, ax=ax1, label='Bq/mL')

im2 = ax2.imshow(suv_map, cmap='hot', vmin=0, vmax=10)
ax2.set_title('SUV Map')
plt.colorbar(im2, ax=ax2, label='SUV')

for ax in (ax1, ax2):
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# Report key values
print(f'Background SUV: {suv_map[~tumor & ~inflammation].mean():.2f} '
      f'(+/- {suv_map[~tumor & ~inflammation].std():.2f})')
print(f'Tumor SUVmax:   {suv_map[tumor].max():.2f}')
print(f'Tumor SUVmean:  {suv_map[tumor].mean():.2f}')
print(f'Inflammation:   {suv_map[inflammation].mean():.2f}')
print(f'\nClinical rule of thumb: SUV > 2.5 is suspicious for malignancy in FDG-PET')

In [ ]:
# ============================================================
# TPD (Total Perfusion Deficit)
# ============================================================
# TPD quantifies the extent and severity of perfusion abnormality
# in cardiac SPECT. It is the standard quantitative metric at
# Cedars-Sinai and many clinical centers.
#
# Concept: compare each segment of the patient's polar map to a
# normal database. Sum up all segments that fall below normal.
#
# TPD = sum over all pixels of max(0, lower_normal_limit - patient_value)
#     / sum over all pixels of lower_normal_limit
#     * 100%
#
# Interpretation:
#   TPD < 5%: Normal
#   TPD 5-10%: Mildly abnormal
#   TPD 10-20%: Moderately abnormal  
#   TPD > 20%: Severely abnormal

def compute_tpd(patient_polar, normal_mean, normal_std, threshold_sd=2.5):
    """
    Compute Total Perfusion Deficit from polar map data.
    
    Parameters
    ----------
    patient_polar : ndarray
        Patient's polar map (perfusion values, 0-100%).
    normal_mean : ndarray
        Mean perfusion from normal database (same shape).
    normal_std : ndarray
        Standard deviation from normal database.
    threshold_sd : float
        Number of SDs below mean to define "abnormal" (default 2.5).
    
    Returns
    -------
    tpd : float
        Total perfusion deficit (0-100%).
    deficit_map : ndarray
        Per-pixel deficit (0 = normal, positive = abnormal).
    """
    # Lower limit of normal for each pixel
    lower_limit = normal_mean - threshold_sd * normal_std
    lower_limit = np.maximum(lower_limit, 0)
    
    # Deficit: how far below normal (clipped at 0)
    deficit_map = np.maximum(lower_limit - patient_polar, 0)
    
    # TPD: total deficit as percentage of total possible
    tpd = np.sum(deficit_map) / np.sum(lower_limit + 1e-10) * 100.0
    
    return tpd, deficit_map


# --- Demo with synthetic polar maps ---
np.random.seed(42)

# Create a synthetic polar map (17-segment model approximation)
# We use a 2D grid as a simplified polar map
n_radial, n_angular = 20, 36  # radial x angular bins
r = np.linspace(0, 1, n_radial)
a = np.linspace(0, 2*np.pi, n_angular, endpoint=False)
R, A = np.meshgrid(r, a, indexing='ij')

# Normal database: perfusion ~ 80-100%, slightly lower at apex
normal_mean = 90 - 10 * R  # apex (center) slightly lower
normal_std = 5 * np.ones_like(normal_mean)

# Patient 1: Normal
patient_normal = normal_mean + np.random.normal(0, 3, normal_mean.shape)
patient_normal = np.clip(patient_normal, 0, 100)

# Patient 2: LAD territory defect (anterior wall)
patient_lad = normal_mean + np.random.normal(0, 3, normal_mean.shape)
# LAD territory: anterior angles (roughly 315-45 degrees)
lad_mask = ((A > 5*np.pi/4) | (A < np.pi/4)) & (R > 0.2)
patient_lad[lad_mask] -= 30  # significant deficit
patient_lad = np.clip(patient_lad, 0, 100)

# Patient 3: Multi-vessel disease
patient_mvd = normal_mean + np.random.normal(0, 3, normal_mean.shape)
patient_mvd[lad_mask] -= 25
rca_mask = ((A > np.pi/2) & (A < 3*np.pi/4)) & (R > 0.3)
patient_mvd[rca_mask] -= 35
patient_mvd = np.clip(patient_mvd, 0, 100)

# Compute TPD for each
patients = [patient_normal, patient_lad, patient_mvd]
patient_names = ['Normal', 'LAD Defect', 'Multi-Vessel Disease']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for i, (patient, name) in enumerate(zip(patients, patient_names)):
    tpd, deficit = compute_tpd(patient, normal_mean, normal_std)
    
    # Polar map (as image)
    axes[0, i].imshow(patient, cmap='hot', vmin=0, vmax=100, aspect='auto')
    axes[0, i].set_title(f'{name}\nPerfusion Polar Map')
    axes[0, i].set_xlabel('Angular Segment')
    axes[0, i].set_ylabel('Radial (apex -> base)')
    
    # Deficit map
    im = axes[1, i].imshow(deficit, cmap='Reds', vmin=0, vmax=40, aspect='auto')
    axes[1, i].set_title(f'Deficit Map\nTPD = {tpd:.1f}%')
    axes[1, i].set_xlabel('Angular Segment')
    axes[1, i].set_ylabel('Radial (apex -> base)')

plt.colorbar(im, ax=axes[1, :].tolist(), label='Perfusion Deficit (%)',
             shrink=0.8)
plt.suptitle('Total Perfusion Deficit (TPD) Calculation', fontsize=14)
plt.tight_layout()
plt.show()

print('TPD Results:')
for patient, name in zip(patients, patient_names):
    tpd, _ = compute_tpd(patient, normal_mean, normal_std)
    status = ('Normal' if tpd < 5 else
              'Mildly Abnormal' if tpd < 10 else
              'Moderately Abnormal' if tpd < 20 else
              'Severely Abnormal')
    print(f'  {name:25s}: TPD = {tpd:5.1f}% -> {status}')

## Statistical Validation Metrics for Clinical AI

When you build an AI model for clinical risk prediction, standard ML metrics (accuracy, AUC) are
necessary but **not sufficient**. Regulatory bodies and clinical journals require you to demonstrate:

1. **Discrimination**: Can the model rank patients by risk correctly? (C-statistic)
2. **Reclassification**: Does the new model move patients into more appropriate risk categories? (NRI, IDI)
3. **Calibration**: Are predicted probabilities reliable? (Brier score, calibration curves)

### Why AUC alone is not enough

- AUC measures rank ordering, not calibration. A model can have AUC 0.90 but predict
  probabilities that are systematically too high or too low.
- AUC is insensitive to clinically meaningful improvements. Adding a biomarker that correctly
  reclassifies 10% of patients may barely change the AUC.
- The FDA and clinical guidelines increasingly require **reclassification analysis** (NRI/IDI)
  to demonstrate that a new model actually changes clinical decisions.

In [ ]:
# ============================================================
# Harrell's C-Statistic (Concordance Index)
# ============================================================
# The C-statistic generalizes AUC to survival/risk data.
# For binary outcomes, C-statistic = AUC of ROC curve.
#
# Definition: Among all pairs of patients where one had the
# event and one didn't, what fraction did the model rank correctly?

def concordance_index(y_true, y_pred):
    """
    Harrell's C-statistic (concordance index).
    
    For binary outcomes, this is equivalent to the AUC.
    
    Parameters
    ----------
    y_true : array-like
        True binary outcomes (0/1).
    y_pred : array-like
        Predicted risk scores or probabilities.
    
    Returns
    -------
    c_stat : float
        Concordance index in [0, 1]. 0.5 = random, 1.0 = perfect.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    concordant = 0
    discordant = 0
    tied_risk = 0
    
    for i in range(n):
        for j in range(i + 1, n):
            if y_true[i] != y_true[j]:  # only consider discordant pairs
                if (y_true[i] > y_true[j] and y_pred[i] > y_pred[j]) or \
                   (y_true[j] > y_true[i] and y_pred[j] > y_pred[i]):
                    concordant += 1
                elif y_pred[i] == y_pred[j]:
                    tied_risk += 1
                else:
                    discordant += 1
    
    total = concordant + discordant + tied_risk
    if total == 0:
        return 0.5
    # Ties count as half-concordant
    return (concordant + 0.5 * tied_risk) / total


# --- Quick validation against sklearn ---
np.random.seed(42)
y_test = np.random.binomial(1, 0.3, 200)
y_score = y_test * 0.6 + np.random.normal(0, 0.3, 200)

our_c = concordance_index(y_test, y_score)
sklearn_auc = roc_auc_score(y_test, y_score)

print(f'Our C-statistic:  {our_c:.4f}')
print(f'sklearn AUC:      {sklearn_auc:.4f}')
print(f'(These should be very close for binary outcomes.)')

In [ ]:
# ============================================================
# NRI (Net Reclassification Improvement) and
# IDI (Integrated Discrimination Improvement)
# ============================================================
# These metrics answer: "Does the new model improve clinical
# decision-making beyond the old model?"
#
# NRI: Among patients who had events, what fraction were moved
#      to a HIGHER risk category by the new model ("correctly
#      reclassified up")? Among those without events, what
#      fraction were moved DOWN ("correctly reclassified down")?
#
# IDI: The average change in predicted probabilities, separately
#      for events and non-events.

def compute_nri(y_true, p_old, p_new, thresholds=(0.06, 0.20)):
    """
    Compute categorical NRI (Net Reclassification Improvement).
    
    Parameters
    ----------
    y_true : ndarray
        True binary outcomes.
    p_old : ndarray
        Predicted probabilities from old model.
    p_new : ndarray
        Predicted probabilities from new model.
    thresholds : tuple of float
        Risk category boundaries. Default (0.06, 0.20) gives:
        Low (<6%), Intermediate (6-20%), High (>20%).
    
    Returns
    -------
    nri : float
        Net reclassification improvement.
    nri_events : float
        NRI component for event patients.
    nri_nonevents : float
        NRI component for non-event patients.
    reclass_table : dict
        Reclassification counts.
    """
    y_true = np.asarray(y_true, dtype=bool)
    
    def categorize(p):
        cats = np.zeros(len(p), dtype=int)
        for i, thresh in enumerate(thresholds):
            cats[p >= thresh] = i + 1
        return cats
    
    cat_old = categorize(p_old)
    cat_new = categorize(p_new)
    
    # Events: fraction moved up minus fraction moved down
    events = y_true
    n_events = np.sum(events)
    up_events = np.sum((cat_new > cat_old) & events)
    down_events = np.sum((cat_new < cat_old) & events)
    nri_events = (up_events - down_events) / n_events
    
    # Non-events: fraction moved down minus fraction moved up
    nonevents = ~y_true
    n_nonevents = np.sum(nonevents)
    up_nonevents = np.sum((cat_new > cat_old) & nonevents)
    down_nonevents = np.sum((cat_new < cat_old) & nonevents)
    nri_nonevents = (down_nonevents - up_nonevents) / n_nonevents
    
    nri = nri_events + nri_nonevents
    
    # Build reclassification table
    n_cats = len(thresholds) + 1
    reclass_table = {
        'events': np.zeros((n_cats, n_cats), dtype=int),
        'nonevents': np.zeros((n_cats, n_cats), dtype=int),
    }
    for i in range(len(y_true)):
        if y_true[i]:
            reclass_table['events'][cat_old[i], cat_new[i]] += 1
        else:
            reclass_table['nonevents'][cat_old[i], cat_new[i]] += 1
    
    return nri, nri_events, nri_nonevents, reclass_table


def compute_idi(y_true, p_old, p_new):
    """
    Compute IDI (Integrated Discrimination Improvement).
    
    IDI = (mean(p_new|event) - mean(p_old|event))
        - (mean(p_new|nonevent) - mean(p_old|nonevent))
    
    Positive IDI means the new model better separates events
    from non-events.
    """
    y_true = np.asarray(y_true, dtype=bool)
    
    # Change in mean predicted probability for events
    delta_events = np.mean(p_new[y_true]) - np.mean(p_old[y_true])
    
    # Change in mean predicted probability for non-events
    delta_nonevents = np.mean(p_new[~y_true]) - np.mean(p_old[~y_true])
    
    # IDI: events should go UP, non-events should go DOWN
    idi = delta_events - delta_nonevents
    
    return idi, delta_events, delta_nonevents


# --- Demo: Generate synthetic cardiac cohort ---
np.random.seed(42)
n_patients = 1000

# Clinical features
age = np.random.normal(62, 10, n_patients).clip(30, 90)
sex = np.random.binomial(1, 0.55, n_patients)  # 1=male
diabetes = np.random.binomial(1, 0.25, n_patients)
hypertension = np.random.binomial(1, 0.40, n_patients)

# Imaging features (these are what our AI model adds)
stress_tpd = np.random.exponential(5, n_patients).clip(0, 40)
cac_score = np.random.exponential(100, n_patients).clip(0, 2000)
ef = np.random.normal(55, 10, n_patients).clip(15, 75)  # ejection fraction
phase_entropy = np.random.normal(50, 15, n_patients).clip(0, 100)  # dyssynchrony

# Generate true outcomes (MACE: major adverse cardiac events)
# True risk depends on ALL features (clinical + imaging)
logit = (-4.0
         + 0.04 * (age - 60)
         + 0.3 * sex
         + 0.5 * diabetes
         + 0.3 * hypertension
         + 0.08 * stress_tpd
         + 0.002 * cac_score
         - 0.03 * ef
         + 0.01 * phase_entropy)
true_prob = 1 / (1 + np.exp(-logit))
y_true = np.random.binomial(1, true_prob)

print(f'Cohort: {n_patients} patients, {y_true.sum()} events '
      f'({100*y_true.mean():.1f}% event rate)')

# --- Old model: logistic regression on clinical features only ---
X_clinical = np.column_stack([age, sex, diabetes, hypertension])

from sklearn.preprocessing import StandardScaler
scaler_clin = StandardScaler()
X_clin_scaled = scaler_clin.fit_transform(X_clinical)

lr_old = LogisticRegression(random_state=42, max_iter=1000)
lr_old.fit(X_clin_scaled, y_true)
p_old = lr_old.predict_proba(X_clin_scaled)[:, 1]

# --- New model: logistic regression adding imaging features ---
X_full = np.column_stack([age, sex, diabetes, hypertension,
                          stress_tpd, cac_score, ef, phase_entropy])
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_full)

lr_new = LogisticRegression(random_state=42, max_iter=1000)
lr_new.fit(X_full_scaled, y_true)
p_new = lr_new.predict_proba(X_full_scaled)[:, 1]

# Compute NRI and IDI
nri, nri_ev, nri_ne, reclass = compute_nri(y_true, p_old, p_new)
idi, idi_ev, idi_ne = compute_idi(y_true, p_old, p_new)

print(f'\n--- Reclassification Analysis ---')
print(f'NRI (total):      {nri:+.3f}  (positive = new model is better)')
print(f'  NRI (events):   {nri_ev:+.3f}  (events reclassified upward)')
print(f'  NRI (nonevents):{nri_ne:+.3f}  (nonevents reclassified downward)')
print(f'\nIDI:              {idi:+.4f}  (positive = better discrimination)')
print(f'  Events delta:   {idi_ev:+.4f}  (events: predicted prob increased)')
print(f'  Nonevents delta:{idi_ne:+.4f}  (nonevents: predicted prob should decrease)')

# --- Reclassification Table ---
cat_names = ['Low (<6%)', 'Mid (6-20%)', 'High (>20%)']
print(f'\n--- Reclassification Table (Events) ---')
print(f'{"":>15s}  New: {"".join(f"{c:>12s}" for c in cat_names)}')
for i, old_cat in enumerate(cat_names):
    row = '  '.join(f'{reclass["events"][i, j]:>10d}' for j in range(3))
    print(f'Old: {old_cat:>10s}  {row}')

print(f'\n--- Reclassification Table (Non-Events) ---')
print(f'{"":>15s}  New: {"".join(f"{c:>12s}" for c in cat_names)}')
for i, old_cat in enumerate(cat_names):
    row = '  '.join(f'{reclass["nonevents"][i, j]:>10d}' for j in range(3))
    print(f'Old: {old_cat:>10s}  {row}')

In [ ]:
# ============================================================
# Brier Score and Calibration Curve
# ============================================================
# Brier score = mean squared error of predicted probabilities.
# Lower is better. Range: 0 (perfect) to 1.
#
# Calibration: if a model predicts 30% risk for a group of
# patients, ~30% of them should actually have events.

def brier_score(y_true, y_pred):
    """Brier score: mean squared error of predicted probabilities."""
    return np.mean((y_pred - y_true) ** 2)


# Compute for both models
bs_old = brier_score(y_true, p_old)
bs_new = brier_score(y_true, p_new)

# Validate against sklearn
bs_old_sk = brier_score_loss(y_true, p_old)
bs_new_sk = brier_score_loss(y_true, p_new)

print(f'Brier Score (old model): {bs_old:.4f}  (sklearn: {bs_old_sk:.4f})')
print(f'Brier Score (new model): {bs_new:.4f}  (sklearn: {bs_new_sk:.4f})')
print(f'Improvement: {bs_old - bs_new:.4f} (lower = better)\n')

# --- Calibration Curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for ax, p_model, name, color in [
    (ax1, p_old, 'Old Model (Clinical Only)', 'steelblue'),
    (ax2, p_new, 'New Model (Clinical + Imaging)', 'crimson'),
]:
    # Compute calibration curve
    prob_true, prob_pred = calibration_curve(y_true, p_model, n_bins=10,
                                             strategy='uniform')
    
    bs = brier_score(y_true, p_model)
    auc = roc_auc_score(y_true, p_model)
    
    # Perfect calibration line
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
    
    # Model calibration
    ax.plot(prob_pred, prob_true, 's-', color=color, linewidth=2,
            markersize=8, label=f'{name}')
    
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Observed Event Rate')
    ax.set_title(f'{name}\nBrier={bs:.4f}, AUC={auc:.3f}')
    ax.legend(loc='upper left')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('A well-calibrated model hugs the diagonal.')
print('Points above the diagonal = model underestimates risk.')
print('Points below the diagonal = model overestimates risk.')

In [ ]:
# ============================================================
# Full Pipeline: XGBoost Cardiac Risk Prediction
# ============================================================
# End-to-end demo comparing baseline (logistic regression on
# clinical features) vs. enhanced (XGBoost with imaging features).

np.random.seed(42)

# --- Generate a fresh synthetic cohort (train/test split) ---
n = 1000

# Features
features = {
    'age':            np.random.normal(62, 10, n).clip(30, 90),
    'sex':            np.random.binomial(1, 0.55, n).astype(float),
    'diabetes':       np.random.binomial(1, 0.25, n).astype(float),
    'hypertension':   np.random.binomial(1, 0.40, n).astype(float),
    'stress_tpd':     np.random.exponential(5, n).clip(0, 40),
    'cac_score':      np.random.exponential(100, n).clip(0, 2000),
    'ef':             np.random.normal(55, 10, n).clip(15, 75),
    'phase_entropy':  np.random.normal(50, 15, n).clip(0, 100),
}

# True outcome (MACE within 3 years)
logit = (-4.0
         + 0.04 * (features['age'] - 60)
         + 0.3 * features['sex']
         + 0.5 * features['diabetes']
         + 0.3 * features['hypertension']
         + 0.08 * features['stress_tpd']
         + 0.002 * features['cac_score']
         - 0.03 * features['ef']
         + 0.01 * features['phase_entropy']
         + 0.05 * features['stress_tpd'] * (features['ef'] < 45)  # interaction
         )
true_prob = 1 / (1 + np.exp(-logit))
y = np.random.binomial(1, true_prob)

# Build feature matrices
clinical_cols = ['age', 'sex', 'diabetes', 'hypertension']
imaging_cols = ['stress_tpd', 'cac_score', 'ef', 'phase_entropy']
all_cols = clinical_cols + imaging_cols

X_clin = np.column_stack([features[c] for c in clinical_cols])
X_all = np.column_stack([features[c] for c in all_cols])

# Train/test split
X_clin_tr, X_clin_te, y_tr, y_te = train_test_split(
    X_clin, y, test_size=0.3, random_state=42, stratify=y
)
X_all_tr, X_all_te = train_test_split(
    X_all, test_size=0.3, random_state=42
)[:2]  # same split

print(f'Training: {len(y_tr)} patients ({y_tr.sum()} events, {100*y_tr.mean():.1f}%)')
print(f'Testing:  {len(y_te)} patients ({y_te.sum()} events, {100*y_te.mean():.1f}%)')

# --- Model 1: Baseline logistic regression (clinical only) ---
scaler_clin = StandardScaler()
X_clin_tr_s = scaler_clin.fit_transform(X_clin_tr)
X_clin_te_s = scaler_clin.transform(X_clin_te)

baseline = LogisticRegression(random_state=42, max_iter=1000)
baseline.fit(X_clin_tr_s, y_tr)
p_baseline = baseline.predict_proba(X_clin_te_s)[:, 1]

# --- Model 2: XGBoost with all features ---
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False,
)
xgb_model.fit(X_all_tr, y_tr, verbose=False)
p_enhanced = xgb_model.predict_proba(X_all_te)[:, 1]

# ============================================================
# Comprehensive Comparison
# ============================================================

# C-statistics
c_base = roc_auc_score(y_te, p_baseline)
c_enh = roc_auc_score(y_te, p_enhanced)

# NRI and IDI
nri_val, nri_ev, nri_ne, reclass = compute_nri(y_te, p_baseline, p_enhanced)
idi_val, idi_ev, idi_ne = compute_idi(y_te, p_baseline, p_enhanced)

# Brier scores
bs_base = brier_score(y_te, p_baseline)
bs_enh = brier_score(y_te, p_enhanced)

# Print results
print('\n' + '='*60)
print('CARDIAC RISK PREDICTION: MODEL COMPARISON')
print('='*60)
print(f'{"Metric":<25s} {"Baseline (LR)":>15s} {"Enhanced (XGB)":>15s} {"Delta":>10s}')
print('-'*65)
print(f'{"C-statistic (AUC)":<25s} {c_base:>15.3f} {c_enh:>15.3f} {c_enh-c_base:>+10.3f}')
print(f'{"Brier Score":<25s} {bs_base:>15.4f} {bs_enh:>15.4f} {bs_enh-bs_base:>+10.4f}')
print(f'{"NRI (total)":<25s} {"--":>15s} {nri_val:>15.3f} {"":>10s}')
print(f'{"  NRI (events)":<25s} {"--":>15s} {nri_ev:>15.3f} {"":>10s}')
print(f'{"  NRI (non-events)":<25s} {"--":>15s} {nri_ne:>15.3f} {"":>10s}')
print(f'{"IDI":<25s} {"--":>15s} {idi_val:>15.4f} {"":>10s}')
print('='*65)

# --- Feature Importance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XGBoost feature importance
importances = xgb_model.feature_importances_
sorted_idx = np.argsort(importances)
axes[0].barh(range(len(all_cols)), importances[sorted_idx], color='steelblue')
axes[0].set_yticks(range(len(all_cols)))
axes[0].set_yticklabels([all_cols[i] for i in sorted_idx])
axes[0].set_xlabel('Feature Importance (gain)')
axes[0].set_title('XGBoost Feature Importances')

# Calibration comparison
for p_model, name, color, marker in [
    (p_baseline, 'Baseline (LR)', 'steelblue', 'o'),
    (p_enhanced, 'Enhanced (XGB)', 'crimson', 's'),
]:
    prob_true, prob_pred = calibration_curve(y_te, p_model, n_bins=8,
                                             strategy='uniform')
    axes[1].plot(prob_pred, prob_true, f'{marker}-', color=color,
                linewidth=2, markersize=8, label=name)

axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Observed Event Rate')
axes[1].set_title('Calibration Comparison')
axes[1].legend()
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Reclassification Table ---
cat_names = ['Low (<6%)', 'Mid (6-20%)', 'High (>20%)']
print('\n--- Reclassification Table (Events, n={}) ---'.format(y_te.sum()))
header = f'{"":>15s} | ' + ' | '.join(f'New:{c:>10s}' for c in cat_names)
print(header)
print('-' * len(header))
for i, old_cat in enumerate(cat_names):
    row = ' | '.join(f'{reclass["events"][i,j]:>14d}' for j in range(3))
    print(f'Old:{old_cat:>10s} | {row}')

print('\n--- Reclassification Table (Non-Events, n={}) ---'.format(
    (1 - y_te).sum()))
header = f'{"":>15s} | ' + ' | '.join(f'New:{c:>10s}' for c in cat_names)
print(header)
print('-' * len(header))
for i, old_cat in enumerate(cat_names):
    row = ' | '.join(f'{reclass["nonevents"][i,j]:>14d}' for j in range(3))
    print(f'Old:{old_cat:>10s} | {row}')

print('\nInterpretation:')
print('  - Positive NRI: new model correctly reclassifies more patients.')
print('  - Events moving UP (to higher risk) = good.')
print('  - Non-events moving DOWN (to lower risk) = good.')
print('  - This is the analysis FDA and journals want to see.')

---

## Summary

### Reconstruction

| Method | Speed | Quality | Noise Handling | Clinical Use |
|--------|-------|---------|---------------|--------------|
| **FBP** | Fastest | Good with clean data | Poor (amplifies noise) | CT (standard) |
| **MLEM** | Slow (30-100 iter) | Excellent | Good (Poisson model) | Research |
| **OSEM** | Fast (2-3 iter x 8-16 sub) | Very good | Good | PET/SPECT (clinical standard) |

Key takeaway: **OSEM is to MLEM what SGD is to full-batch gradient descent.** Same idea, same
trade-offs, same acceleration.

### Clinical Metrics

| Metric | What it measures | When to use |
|--------|-----------------|-------------|
| **Dice** | Segmentation overlap | Any segmentation task |
| **SUV** | Quantitative tracer uptake | PET imaging |
| **TPD** | Perfusion abnormality extent | Cardiac SPECT |
| **C-statistic** | Discrimination (rank ordering) | Risk prediction |
| **NRI** | Reclassification improvement | Model comparison |
| **IDI** | Discrimination improvement | Model comparison |
| **Brier Score** | Calibration + discrimination | Probability predictions |

Key takeaway: **AUC alone is never sufficient for clinical AI validation.** You must show
reclassification improvement (NRI/IDI) and calibration (Brier/calibration curves).

### Next Notebooks

- **03a**: CNN architectures for medical imaging (U-Net, attention, residual connections)
- **03b**: Transfer learning and domain adaptation for small medical datasets
- **03c**: Vision transformers and hybrid architectures for volumetric data